# Preprocessing và validation dữ liệu daily

Luồng v0: `raw VNDIRECT snapshot -> adapter -> contract validator -> fixture cố định`.

Validator không sort, fill, forward-fill hoặc sửa OHLC. Input sai bị từ chối bằng `DataValidationError` có danh sách lỗi dạng dictionary.


In [1]:
from __future__ import annotations

import hashlib
import json
import math
from collections.abc import Mapping, Sequence
from copy import deepcopy
from datetime import date, datetime, timezone
from numbers import Real
from pathlib import Path
from typing import Any

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "data":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / "vndirect"
FIXTURE_DIR = PROJECT_ROOT / "tests" / "fixtures"
EXPECTED_TIMEFRAME = "1D"


In [2]:
Issue = dict[str, Any]

class DataValidationError(ValueError):
    """Một hoặc nhiều lỗi vi phạm Data Contract."""
    def __init__(self, issues: Sequence[Issue]):
        self.issues = list(issues)
        super().__init__("; ".join(f"{x['code']}: {x['message']}" for x in self.issues))

    def to_dict(self) -> dict[str, Any]:
        return {"error": "data_validation_failed", "issues": self.issues}

def _issue(code: str, message: str, *, row: int | None = None, field: str | None = None, value: Any = None) -> Issue:
    result: Issue = {"code": code, "message": message}
    if row is not None:
        result["row"] = row
    if field is not None:
        result["field"] = field
    if value is not None:
        result["value"] = value
    return result

def _raise_if_invalid(issues: Sequence[Issue]) -> None:
    if issues:
        raise DataValidationError(issues)

def _is_missing(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, str):
        return not value.strip()
    return isinstance(value, Real) and not isinstance(value, bool) and math.isnan(value)

def _is_finite_number(value: Any) -> bool:
    return isinstance(value, Real) and not isinstance(value, bool) and math.isfinite(value)

def _parse_trading_date(value: Any) -> date | None:
    if isinstance(value, datetime):
        return value.date()
    if isinstance(value, date):
        return value
    if not isinstance(value, str):
        return None
    try:
        parsed = date.fromisoformat(value)
        return parsed if parsed.isoformat() == value else None
    except ValueError:
        return None


In [3]:
METADATA_REQUIRED_FIELDS = (
    "dataset_id", "dataset_version", "content_hash", "source",
    "extracted_at", "timeframe", "timezone", "price_unit",
    "currency", "price_adjustment", "volume_adjustment",
    "corporate_action_policy",
)
BAR_REQUIRED_FIELDS = {
    "HPG": ("symbol", "trading_date", "open", "high", "low", "close", "volume"),
    "VNINDEX": ("symbol", "trading_date", "close"),
}

def _metadata_issues(metadata: Any) -> list[Issue]:
    if not isinstance(metadata, Mapping):
        return [_issue("INVALID_METADATA", "metadata phải là object/dictionary")]
    issues: list[Issue] = []
    for field in METADATA_REQUIRED_FIELDS:
        if field not in metadata:
            issues.append(_issue("REQUIRED_FIELD_MISSING", f"Thiếu metadata field: {field}", field=field))
        elif _is_missing(metadata[field]):
            issues.append(_issue("MISSING_VALUE", f"Metadata {field} không được null/rỗng", field=field))
    if "timeframe" in metadata and not _is_missing(metadata["timeframe"]) and metadata["timeframe"] != EXPECTED_TIMEFRAME:
        issues.append(_issue("INVALID_TIMEFRAME", f"Strategy v0 chỉ nhận timeframe {EXPECTED_TIMEFRAME}", field="timeframe", value=metadata.get("timeframe")))
    return issues

def validate_metadata(metadata: Mapping[str, Any]) -> dict[str, Any]:
    _raise_if_invalid(_metadata_issues(metadata))
    return dict(metadata)

def _bar_issues(bars: Any, symbol: str) -> list[Issue]:
    if symbol not in BAR_REQUIRED_FIELDS:
        return [_issue("UNSUPPORTED_SYMBOL", "v0 chỉ hỗ trợ HPG hoặc VNINDEX", value=symbol)]
    if not isinstance(bars, Sequence) or isinstance(bars, (str, bytes, bytearray)):
        return [_issue("INVALID_BARS", "bars phải là một array các object")]
    if not bars:
        return [_issue("EMPTY_DATASET", f"Không có daily bar cho {symbol}")]

    issues: list[Issue] = []
    seen_dates: dict[date, int] = {}
    previous_date: date | None = None
    for row_number, bar in enumerate(bars):
        if not isinstance(bar, Mapping):
            issues.append(_issue("INVALID_ROW", "Mỗi bar phải là object/dictionary", row=row_number))
            continue
        for field in BAR_REQUIRED_FIELDS[symbol]:
            if field not in bar:
                issues.append(_issue("REQUIRED_FIELD_MISSING", f"Thiếu bar field: {field}", row=row_number, field=field))
            elif _is_missing(bar[field]):
                issues.append(_issue("MISSING_VALUE", f"{field} không được null/rỗng/NaN", row=row_number, field=field))
        if bar.get("symbol") != symbol:
            issues.append(_issue("INVALID_SYMBOL", f"Expected symbol {symbol}", row=row_number, field="symbol", value=bar.get("symbol")))
        parsed_date = _parse_trading_date(bar.get("trading_date"))
        if parsed_date is None and not _is_missing(bar.get("trading_date")):
            issues.append(_issue("INVALID_TRADING_DATE", "trading_date phải có dạng YYYY-MM-DD", row=row_number, field="trading_date", value=bar.get("trading_date")))
        elif parsed_date is not None:
            if parsed_date in seen_dates:
                issues.append(_issue("DUPLICATE_TRADING_DATE", f"trading_date trùng row {seen_dates[parsed_date]}", row=row_number, field="trading_date", value=parsed_date.isoformat()))
            else:
                seen_dates[parsed_date] = row_number
            if previous_date is not None and parsed_date < previous_date:
                issues.append(_issue("OUT_OF_ORDER_TRADING_DATE", "trading_date phải tăng dần; validator không tự sort", row=row_number, field="trading_date", value=parsed_date.isoformat()))
            previous_date = parsed_date

        numeric_fields = ("open", "high", "low", "close", "volume") if symbol == "HPG" else ("close",)
        for field in numeric_fields:
            value = bar.get(field)
            if not _is_missing(value) and not _is_finite_number(value):
                issues.append(_issue("INVALID_NUMBER", f"{field} phải là finite number", row=row_number, field=field, value=value))

        if symbol == "HPG":
            prices = {name: bar.get(name) for name in ("open", "high", "low", "close")}
            if all(_is_finite_number(x) for x in prices.values()):
                for field, value in prices.items():
                    if value <= 0:
                        issues.append(_issue("NON_POSITIVE_PRICE", f"{field} phải > 0", row=row_number, field=field, value=value))
                if prices["high"] < max(prices["open"], prices["close"]):
                    issues.append(_issue("INVALID_OHLC_HIGH", "high phải >= max(open, close)", row=row_number, field="high", value=prices["high"]))
                if prices["low"] > min(prices["open"], prices["close"]):
                    issues.append(_issue("INVALID_OHLC_LOW", "low phải <= min(open, close)", row=row_number, field="low", value=prices["low"]))
                if prices["high"] < prices["low"]:
                    issues.append(_issue("INVALID_OHLC_RANGE", "high phải >= low", row=row_number))
            volume = bar.get("volume")
            if _is_finite_number(volume) and volume < 0:
                issues.append(_issue("NEGATIVE_VOLUME", "volume phải >= 0", row=row_number, field="volume", value=volume))
        else:
            close = bar.get("close")
            if _is_finite_number(close) and close <= 0:
                issues.append(_issue("NON_POSITIVE_PRICE", "close phải > 0", row=row_number, field="close", value=close))

    return issues

def validate_daily_bars(bars: Sequence[Mapping[str, Any]], symbol: str) -> list[dict[str, Any]]:
    _raise_if_invalid(_bar_issues(bars, symbol))
    return [dict(bar) for bar in bars]

def _alignment_issues(hpg_bars: Any, vnindex_bars: Any) -> list[Issue]:
    if not isinstance(hpg_bars, Sequence) or not isinstance(vnindex_bars, Sequence):
        return []
    hpg_dates = {_parse_trading_date(bar.get("trading_date")) for bar in hpg_bars if isinstance(bar, Mapping)}
    vnindex_dates = {_parse_trading_date(bar.get("trading_date")) for bar in vnindex_bars if isinstance(bar, Mapping)}
    missing_dates = sorted(x.isoformat() for x in hpg_dates - vnindex_dates if x is not None)
    if not missing_dates:
        return []
    return [_issue("MISSING_VNINDEX_DATE", "Thiếu VNINDEX Close cho ngày HPG; không được fill ngầm", field="trading_date", value=missing_dates)]

def validate_dataset(metadata: Mapping[str, Any], hpg_bars: Sequence[Mapping[str, Any]], vnindex_bars: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    issues = [*_metadata_issues(metadata), *_bar_issues(hpg_bars, "HPG"), *_bar_issues(vnindex_bars, "VNINDEX"), *_alignment_issues(hpg_bars, vnindex_bars)]
    _raise_if_invalid(issues)
    return {"metadata": dict(metadata), "hpg": [dict(x) for x in hpg_bars], "vnindex": [dict(x) for x in vnindex_bars]}


## Adapter VNDIRECT dchart

Raw payload dùng `t/o/h/l/c/v`. Adapter chỉ map field và Unix timestamp UTC sang `trading_date`; không sort hay điền dữ liệu. `resolution=D` được map explicit thành contract timeframe `1D`.


In [4]:
DCHART_FIELDS = {
    "HPG": {"t": "trading_date", "o": "open", "h": "high", "l": "low", "c": "close", "v": "volume"},
    "VNINDEX": {"t": "trading_date", "c": "close"},
}

def dchart_resolution_to_timeframe(resolution: Any) -> str:
    if resolution != "D":
        raise DataValidationError([_issue("INVALID_TIMEFRAME", "VNDIRECT input cho strategy v0 phải dùng resolution=D", field="resolution", value=resolution)])
    return EXPECTED_TIMEFRAME

def adapt_dchart_payload(payload: Mapping[str, Any], *, symbol: str, resolution: str) -> list[dict[str, Any]]:
    dchart_resolution_to_timeframe(resolution)
    if symbol not in DCHART_FIELDS:
        raise DataValidationError([_issue("UNSUPPORTED_SYMBOL", "v0 chỉ hỗ trợ HPG hoặc VNINDEX", value=symbol)])
    if not isinstance(payload, Mapping):
        raise DataValidationError([_issue("INVALID_PAYLOAD", "dchart payload phải là object/dictionary")])
    issues: list[Issue] = []
    if "s" in payload and payload["s"] != "ok":
        issues.append(_issue("SOURCE_STATUS_NOT_OK", "dchart status không phải ok", field="s", value=payload["s"]))
    arrays: dict[str, Sequence[Any]] = {}
    for raw_field in DCHART_FIELDS[symbol]:
        if raw_field not in payload:
            issues.append(_issue("REQUIRED_FIELD_MISSING", f"Thiếu raw field: {raw_field}", field=raw_field))
        elif not isinstance(payload[raw_field], Sequence) or isinstance(payload[raw_field], (str, bytes, bytearray)):
            issues.append(_issue("INVALID_ARRAY", f"Raw field {raw_field} phải là array", field=raw_field))
        else:
            arrays[raw_field] = payload[raw_field]
    lengths = {field: len(values) for field, values in arrays.items()}
    if lengths and len(set(lengths.values())) != 1:
        issues.append(_issue("ARRAY_LENGTH_MISMATCH", "Các array dchart phải có cùng số phần tử", value=lengths))
    if lengths and next(iter(lengths.values())) == 0:
        issues.append(_issue("EMPTY_DATASET", f"Không có raw bar cho {symbol}"))
    _raise_if_invalid(issues)

    bars: list[dict[str, Any]] = []
    timestamp_issues: list[Issue] = []
    for row_number in range(next(iter(lengths.values()))):
        raw_timestamp = arrays["t"][row_number]
        trading_date: str | None = None
        if not _is_finite_number(raw_timestamp):
            timestamp_issues.append(_issue("INVALID_TIMESTAMP", "t phải là Unix seconds hữu hạn", row=row_number, field="t", value=raw_timestamp))
        elif raw_timestamp % 86400 != 0:
            timestamp_issues.append(_issue("INVALID_DAILY_TIMESTAMP", "Daily timestamp phải ở 00:00 UTC theo profile đã chốt", row=row_number, field="t", value=raw_timestamp))
        else:
            try:
                trading_date = datetime.fromtimestamp(raw_timestamp, tz=timezone.utc).date().isoformat()
            except (OverflowError, OSError, ValueError):
                timestamp_issues.append(_issue("INVALID_TIMESTAMP", "t nằm ngoài range hỗ trợ", row=row_number, field="t", value=raw_timestamp))
        bar: dict[str, Any] = {"symbol": symbol, "trading_date": trading_date}
        for raw_field, contract_field in DCHART_FIELDS[symbol].items():
            if raw_field != "t":
                bar[contract_field] = arrays[raw_field][row_number]
        bars.append(bar)
    _raise_if_invalid(timestamp_issues)
    return validate_daily_bars(bars, symbol)


In [5]:
def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)

def write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(value, file, ensure_ascii=False, indent=2, allow_nan=False)
        file.write("\n")

def content_hash(paths: Sequence[Path]) -> str:
    digest = hashlib.sha256()
    for path in sorted(paths, key=lambda item: item.name):
        digest.update(path.name.encode("utf-8") + b"\0")
        with path.open("rb") as file:
            for chunk in iter(lambda: file.read(1024 * 1024), b""):
                digest.update(chunk)
    return digest.hexdigest()

def select_aligned_fixture(hpg_bars: Sequence[Mapping[str, Any]], vnindex_bars: Sequence[Mapping[str, Any]], *, count: int = 5) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    if count < 1:
        raise ValueError("count phải >= 1")
    vn_by_date = {bar["trading_date"]: dict(bar) for bar in vnindex_bars}
    shared_dates = [bar["trading_date"] for bar in hpg_bars if bar["trading_date"] in vn_by_date][:count]
    if len(shared_dates) < count:
        raise DataValidationError([_issue("INSUFFICIENT_ALIGNED_ROWS", f"Chỉ có {len(shared_dates)} ngày giao nhau; cần {count}")])
    hpg_by_date = {bar["trading_date"]: dict(bar) for bar in hpg_bars}
    return [hpg_by_date[x] for x in shared_dates], [vn_by_date[x] for x in shared_dates]


## Cách lấy input và tạo fixture

Tải **mỗi endpoint một lần** thành raw snapshot, ghi lại thời điểm tải, rồi chạy adapter/validator. Không dùng live URL trong test và không dùng toàn bộ dataset làm fixture.

Chạy từ project root bằng PowerShell:

```powershell
New-Item -ItemType Directory -Force -Path 'data/raw/vndirect'
$hpgUrl = 'https://dchart-api.vndirect.com.vn/dchart/history?resolution=D&symbol=HPG&from=1546300800&to=1704067200'
$vnindexUrl = 'https://dchart-api.vndirect.com.vn/dchart/history?resolution=D&symbol=VNINDEX&from=1546300800&to=1704067200'
Invoke-WebRequest -Uri $hpgUrl -OutFile 'data/raw/vndirect/hpg_2019_2023.json'
Invoke-WebRequest -Uri $vnindexUrl -OutFile 'data/raw/vndirect/vnindex_2019_2023.json'
Get-Date -AsUTC -Format o
```

Lưu `Get-Date` làm `extracted_at`. Metadata nguồn chưa công bố phải ghi `unknown`/limitation, không đoán. `data/` đang Git-ignore; chỉ fixture nhỏ, cố định mới xuất vào `tests/fixtures/`.


In [6]:
# Chạy sau khi tải hai snapshot; bỏ comment và thay extracted_at.
hpg_raw_path = DATA_DIR / "raw_hpg_dchart_2019_2023.json"
vnindex_raw_path = DATA_DIR / "raw_vnindex_dchart_2019_2023.json"
hpg_bars = adapt_dchart_payload(load_json(hpg_raw_path), symbol="HPG", resolution="D")
vnindex_bars = adapt_dchart_payload(load_json(vnindex_raw_path), symbol="VNINDEX", resolution="D")
metadata = load_json(DATA_DIR / "dataset_manifest.json")
assert metadata["content_hash"] == content_hash([hpg_raw_path, vnindex_raw_path])
validate_dataset(metadata, hpg_bars, vnindex_bars)
fixture_hpg, fixture_vnindex = select_aligned_fixture(hpg_bars, vnindex_bars, count=5)
validate_dataset(metadata, fixture_hpg, fixture_vnindex)
write_json(FIXTURE_DIR / "dataset_metadata_valid.json", metadata)
write_json(FIXTURE_DIR / "hpg_daily_valid.json", fixture_hpg)
write_json(FIXTURE_DIR / "vnindex_daily_valid.json", fixture_vnindex)


## Smoke tests không phụ thuộc live API

Các record synthetic chỉ test boundary validator. Fixture integration có provenance vẫn phải trích từ snapshot thật bằng cell trên.


In [7]:
VALID_METADATA = {
    "dataset_id": "validator-smoke-test", "dataset_version": "1",
    "content_hash": "synthetic-not-for-backtest", "source": "synthetic validator boundary",
    "extracted_at": "2026-09-12T00:00:00Z", "timeframe": "1D",
    "timezone": "UTC", "price_unit": "synthetic", "currency": "synthetic",
    "price_adjustment": "synthetic", "volume_adjustment": "synthetic",
    "corporate_action_policy": "synthetic",
}
VALID_HPG = [
    {"symbol": "HPG", "trading_date": "2023-01-03", "open": 18.0, "high": 18.8, "low": 17.9, "close": 18.5, "volume": 1_000_000},
    {"symbol": "HPG", "trading_date": "2023-01-04", "open": 18.6, "high": 19.0, "low": 18.4, "close": 18.9, "volume": 1_200_000},
]
VALID_VNINDEX = [
    {"symbol": "VNINDEX", "trading_date": "2023-01-03", "close": 1043.9},
    {"symbol": "VNINDEX", "trading_date": "2023-01-04", "close": 1046.4},
]

def assert_rejected(expected_code: str, function, *args, **kwargs) -> None:
    try:
        function(*args, **kwargs)
    except DataValidationError as error:
        assert expected_code in {x["code"] for x in error.issues}, (expected_code, error.to_dict())
    else:
        raise AssertionError(f"Expected {expected_code}")

validate_dataset(VALID_METADATA, VALID_HPG, VALID_VNINDEX)
missing_field = deepcopy(VALID_HPG); del missing_field[0]["volume"]
assert_rejected("REQUIRED_FIELD_MISSING", validate_daily_bars, missing_field, "HPG")
assert_rejected("OUT_OF_ORDER_TRADING_DATE", validate_daily_bars, list(reversed(deepcopy(VALID_HPG))), "HPG")
duplicate = deepcopy(VALID_HPG); duplicate[1]["trading_date"] = duplicate[0]["trading_date"]
assert_rejected("DUPLICATE_TRADING_DATE", validate_daily_bars, duplicate, "HPG")
invalid_ohlc = deepcopy(VALID_HPG); invalid_ohlc[0]["high"] = 18.1
assert_rejected("INVALID_OHLC_HIGH", validate_daily_bars, invalid_ohlc, "HPG")
missing_value = deepcopy(VALID_HPG); missing_value[0]["close"] = None
assert_rejected("MISSING_VALUE", validate_daily_bars, missing_value, "HPG")
assert_rejected("INVALID_TIMEFRAME", validate_metadata, {**VALID_METADATA, "timeframe": "1h"})
assert_rejected("MISSING_VNINDEX_DATE", validate_dataset, VALID_METADATA, VALID_HPG, VALID_VNINDEX[:1])
dated_hpg = [{**bar, "trading_date": date.fromisoformat(bar["trading_date"])} for bar in VALID_HPG]
dated_vnindex = [{**bar, "trading_date": date.fromisoformat(bar["trading_date"])} for bar in VALID_VNINDEX[:1]]
assert_rejected("MISSING_VNINDEX_DATE", validate_dataset, VALID_METADATA, dated_hpg, dated_vnindex)
assert_rejected("INVALID_TIMEFRAME", dchart_resolution_to_timeframe, "1")
print("Smoke tests passed: required fields, ordering, duplicate, OHLC, missing value/alignment, timeframe.")


Smoke tests passed: required fields, ordering, duplicate, OHLC, missing value/alignment, timeframe.
